# 04 — Fine-Tuning the Age-Group Classifier

**Project:** AuraVision — Crowd Age Group Majority Classifier  
**Purpose:** fine-tune the pretrained `age_classifier.pth` to reduce overfitting and improve generalisation on the held-out test set.

Key differences from notebook 02:
- The **entire backbone is frozen**; only the classifier head is retrained.
- **Stronger data augmentation** (random-resized crop, perspective distortion, random erasing).
- **Heavier dropout** (0.5 / 0.4) and **10× stronger weight decay** (1e-2).
- **Label smoothing** in the loss function.
- **Test-Time Augmentation (TTA)** averages five augmented predictions at inference.
- **Cosine Annealing with Warm Restarts** resets the learning rate twice to escape local minima.

## Colab setup

In [ ]:
!pip install -q torch torchvision pillow numpy

import os
print("Current working directory:", os.getcwd())
print("Files here:", os.listdir("."))

## Step 0 — Imports

In [ ]:
import os
import copy
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from PIL import Image

## Step 1 — Configuration & Settings

In [ ]:
# Paths
TRAIN_DIR  = "split_dataset/train"          # subfolders: Children/, Adults/, Seniors/
TEST_DIR   = "test_dataset"                 # subfolders with labeled test images
MODEL_PATH = "age_classifier.pth"           # original model — starting point
SAVE_PATH  = "age_classifier_finetuned.pth" # output of this notebook

# Hardware
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {DEVICE}")

# Hyperparameters
NUM_CLASSES = 3    # Children, Adults, Seniors
EPOCHS      = 20
BATCH_SIZE  = 8
LR          = 1e-4

## Step 2 — Data Transforms

Training uses aggressive augmentation to reduce memorisation. Validation and Test-Time Augmentation (TTA) use deterministic or mildly varied crops.

In [ ]:
# Training transforms — stronger than notebook 02 to fight overfitting
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.RandomRotation(20),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.15)),  # hides random patches
])

# Validation / standard inference transform
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# TTA: 5 different crops/flips averaged at inference
tta_transforms = [
    transforms.Compose([
        transforms.Resize(256), transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    transforms.Compose([
        transforms.Resize(256), transforms.RandomCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    transforms.Compose([
        transforms.Resize(256), transforms.CenterCrop(224),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    transforms.Compose([
        transforms.Resize(280), transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    transforms.Compose([
        transforms.Resize(256), transforms.FiveCrop(224),
        transforms.Lambda(lambda crops: crops[0]),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
]

print("Transforms defined.")

## Step 3 — Build Data Loaders

A `WeightedRandomSampler` balances the class distribution during training, so underrepresented classes receive equal attention.

In [ ]:
def build_loaders():
    train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
    test_dataset  = datasets.ImageFolder(TEST_DIR,  transform=val_transform)

    # Weighted sampler for class balance
    class_counts = [0] * NUM_CLASSES
    for _, label in train_dataset.samples:
        class_counts[label] += 1
    weights = [1.0 / class_counts[label] for _, label in train_dataset.samples]
    sampler = torch.utils.data.WeightedRandomSampler(weights, len(weights))

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
    test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

    print(f"Classes : {train_dataset.class_to_idx}")
    print(f"Train   : {len(train_dataset)} images")
    print(f"Test    : {len(test_dataset)} images")

    return train_loader, test_loader, train_dataset.class_to_idx


train_loader, test_loader, class_to_idx = build_loaders()

## Step 4 — Build the Model

We load the weights saved by notebook 02, then **freeze the entire backbone** and only allow the new classifier head to update. This prevents catastrophic forgetting and keeps training fast.

In [ ]:
def build_model():
    model = models.efficientnet_b0(weights=None)

    # Heavier dropout in the head — fights memorisation
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.5),           # was 0.4 in notebook 02
        nn.Linear(in_features, 128),
        nn.ReLU(),
        nn.Dropout(0.4),           # was 0.3 in notebook 02
        nn.Linear(128, NUM_CLASSES),
    )

    # Load weights from the originally trained model
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    print(f"Loaded weights from: {MODEL_PATH}")

    # Freeze the entire backbone — only the classifier head will train
    for param in model.features.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"Trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.1f}%)")

    return model.to(DEVICE)


model = build_model()

## Step 5 — Evaluation Helpers

Two evaluation functions:
- `evaluate` — standard single-pass accuracy on a DataLoader.
- `evaluate_with_tta` — averages predictions from 5 augmented views of each image.

In [ ]:
def evaluate(model, loader):
    """Standard single-pass accuracy on a DataLoader."""
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            _, preds = torch.max(model(inputs), 1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    return correct / total


def evaluate_with_tta(model, test_dir, class_to_idx):
    """Run inference with TTA — averages 5 augmented predictions per image."""
    model.eval()
    correct = total = 0

    for class_name, class_idx in class_to_idx.items():
        class_dir = os.path.join(test_dir, class_name)
        if not os.path.isdir(class_dir):
            continue
        for fname in os.listdir(class_dir):
            if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
            img = Image.open(os.path.join(class_dir, fname)).convert("RGB")

            # Average logits across all TTA transforms
            all_logits = []
            with torch.no_grad():
                for tfm in tta_transforms:
                    tensor = tfm(img).unsqueeze(0).to(DEVICE)
                    all_logits.append(model(tensor))

            avg_logits = torch.stack(all_logits).mean(0)
            pred = avg_logits.argmax(dim=1).item()

            correct += int(pred == class_idx)
            total   += 1

    return correct / total if total > 0 else 0.0


print("Evaluation functions defined.")

## Step 6 — Fine-Tuning Loop

Notable choices:
- **Label smoothing (0.1)** prevents the model from becoming overconfident.
- **AdamW weight decay (1e-2)** is 10× stronger than the original run — the main lever against overfitting.
- **Cosine Annealing with Warm Restarts** (`T_0=10`) resets the learning rate at epochs 10 and 20 to help escape local minima.

In [ ]:
# Loss, optimiser, and scheduler
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.AdamW(
    model.classifier.parameters(),
    lr=LR,
    weight_decay=1e-2    # 10× stronger regularisation than original
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=1
)

# Establish baseline before any fine-tuning
baseline = evaluate(model, test_loader)
best_acc     = baseline
best_weights = copy.deepcopy(model.state_dict())

print(f"Baseline accuracy (no TTA): {baseline:.2%}")
print("-" * 65)

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = correct = total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(inputs), labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(model(inputs), 1)
        correct  += (preds == labels).sum().item()
        total    += labels.size(0)

    scheduler.step()

    train_acc = correct / total
    test_acc  = evaluate(model, test_loader)

    marker = ""
    if test_acc > best_acc:
        best_acc     = test_acc
        best_weights = copy.deepcopy(model.state_dict())
        torch.save(best_weights, SAVE_PATH)
        marker = "  ← saved"

    print(f"Epoch {epoch:02d}/{EPOCHS}  "
          f"loss: {running_loss/total:.4f}  "
          f"train: {train_acc:.2%}  "
          f"test: {test_acc:.2%}{marker}")

## Step 7 — Final Evaluation with TTA

Load the best checkpoint and report both standard and TTA accuracy. A large remaining train/test gap (> 20 %) suggests more training data is needed.

In [ ]:
# Load the best checkpoint
model.load_state_dict(best_weights)

# Evaluate with TTA for the final score
tta_acc = evaluate_with_tta(model, TEST_DIR, class_to_idx)

print("\n" + "=" * 65)
print(f"Best test accuracy (standard) : {best_acc:.2%}")
print(f"Best test accuracy (with TTA) : {tta_acc:.2%}")
print(f"Saved to                      : {SAVE_PATH}")
print("=" * 65)

# Overfitting diagnosis
train_acc_final = evaluate(model, train_loader)
gap = train_acc_final - best_acc
if gap > 0.20:
    print("\n⚠  Train/test gap still large — consider collecting more data.")
    print("   Target: 300+ images per class with diverse contexts.")
else:
    print("\n✓  Overfitting gap reduced. Model is generalising better.")

## Output of this notebook

After running this notebook, you should have:

```text
age_classifier_finetuned.pth   # fine-tuned model weights (best checkpoint)
```

Pass `age_classifier_finetuned.pth` to notebook 03 to generate an updated confusion matrix, or use it directly for inference.